# 

## Celda 0: entorno

In [1]:
# recarga automatica de los .py de src/ al editarlos, sin reiniciar el kernel
%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# validacion: el notebook debe ejecutarse desde pre_proyecto/notebooks/
if not Path("../src/geo_download.py").exists():
    raise FileNotFoundError("Abre este notebook desde pre_proyecto/notebooks/")

# anade ../src a las carpetas donde Python busca modulos
sys.path.insert(0, "../src")

# rutas relativas a la carpeta del notebook
RAW = Path("../data/raw")
PROC = Path("../data/processed")
TABLES = Path("../results/tables")
# crea las carpetas que falten
for folder in (RAW, PROC, TABLES):
    folder.mkdir(parents=True, exist_ok=True)
print("Entorno listo")

Entorno listo


## Celda 1: consultar la serie

In [2]:
# Entrez se importa aqui solo para registrar el correo
from Bio import Entrez
import geo_download as geo

# NCBI exige un correo
Entrez.email = "andcoral@lcg.unam.mx"

ACCESSION = "GSE103441"
summary = geo.get_gse_summary(ACCESSION)
print(summary["Accession"], "|", summary["title"])
print("N muestras:", int(summary["n_samples"]))

GSE103441 | In planta bacterial RNA-Seq with bacterial isolation.
N muestras: 114


## Celda 2: BioProject y tabla de muestras

In [3]:
# {codigo PRJ: True} si existe en NCBI
print("BioProject:", geo.validate_bioproject(summary))

# tabla con gsm y title; valida que sean las 114
samples = geo.get_samples_table(summary)
samples.to_csv(PROC / "GEO_samples.csv", index=False)
samples

BioProject: {'PRJNA401856': True}


,gsm,title
0,GSM2771404,Pto-AvrRpt2_col_A25
1,GSM2771424,Pto-AvrRpt2_col_A47
2,GSM2771470,Pto-AvrRpt2_KB_A81
3,GSM2771407,Pto_col_A26
4,GSM2771444,Pto_col_A58
...,...,...
109,GSM2771461,Pto_Col_A74
110,GSM2771378,Pto-AvrRpt2_sid2_A17
111,GSM2771438,Pto_col_chitin_A54
112,GSM2771455,Pto-AvrRpt2_pad4_A66


## Celda 3: archivos suplementarios

In [4]:
# Imprime lista de archivos suplementarios del GEO
suppl_url, files = geo.list_supplementary_files(summary)
print(suppl_url)
for file_name in files:
    print(" -", file_name)

https://ftp.ncbi.nlm.nih.gov/geo/series/GSE103nnn/GSE103441/suppl/
 - GSE103441_count.txt.gz


## Celda 4: seleccionar muestras

In [5]:
FILE_NAME = "GSE103441_count.txt.gz"

# el nombre debe estar en la lista que devolvio el servidor
if FILE_NAME not in files:
    raise FileNotFoundError(f"{FILE_NAME} no está. Disponibles: {files}")

# Descarga archivo y lo valida
path = geo.download_file(suppl_url, FILE_NAME, RAW)
count_matrix = geo.validate_count_matrix(path)
count_matrix.iloc[:5, :6]

Ya existe: ..\data\raw\GSE103441_count.txt.gz
Genes x muestras: (5618, 114)
¿Todos enteros? True
NaN: 0 | IDs duplicados: 0


,Pto_col_mock_A22,Pto_col_mock_A23,Pto_col_mock_A52,Pto_col_mock_A53,Pto_col_mock_A54,Pto_col_chitin_A52
PSPTOA0002,0,0,0,0,0,0
PSPTOA0004,0,0,0,0,0,0
PSPTOA0005,0,0,0,0,0,0
PSPTOA0007,0,0,0,0,3,0
PSPTOA0008,0,0,0,0,0,0


## Celda 5: seleccionar muestras

In [6]:
import data_prep as prep

selection = prep.select_samples(count_matrix.columns)
print(selection["condition"].value_counts())
# solo las elegidas, para revisarlas a ojo
print(selection[selection["condition"] != "exclude"].to_string())

# Dataset S1: 13 de planta y 2 de cultivo; si no coincide, se detiene
prep.verify_selection(selection, {"plant": 13, "culture": 2})

condition
exclude    99
plant      13
culture     2
Name: count, dtype: int64
          sample condition
11   Pto_col_A13     plant
12   Pto_col_A14     plant
13   Pto_col_A15     plant
14   Pto_col_A17     plant
15   Pto_col_A25     plant
16   Pto_col_A26     plant
17   Pto_col_A42     plant
18   Pto_col_A66     plant
19   Pto_col_A57     plant
20   Pto_col_A58     plant
21   Pto_col_A59     plant
88    Pto_KB_A81   culture
89    Pto_KB_A83   culture
106  Pto_col_A71     plant
110  Pto_col_A74     plant
Selección verificada: {'plant': 13, 'culture': 2}


## Celda 6: separar y validar IDs de gen

In [7]:
# separa cada ID de fila ("PSPTO_0001-descripcion") en locus_tag y descripcion
# y crea locus_key sin guion bajo (PSPTO0001), la clave para unir con GenBank
ids_table = prep.split_id_description(count_matrix.index)

# Valida e imprime
ids_table = prep.split_id_description(count_matrix.index)
prep.validate_gene_ids(ids_table)

# 6 filas al azar para verlas
ids_table.sample(6, random_state=1)

Locus tag con formato PSPTO: 5618 de 5618
Cromosoma: 5481 | Plásmido A: 67 | Plásmido B: 70
Genes con descripción: 5552
Locus tag repetidos: 0


,original_id,locus_tag,description,locus_key
1840,PSPTO_1819-major facilitator superfamily trans...,PSPTO_1819,major facilitator superfamily transporter,PSPTO1819
3198,"PSPTO_3221-ISPsy5, Orf1",PSPTO_3221,"ISPsy5, Orf1",PSPTO3221
5506,PSPTO_5600-F0F1 ATP synthase subunit gamma,PSPTO_5600,F0F1 ATP synthase subunit gamma,PSPTO5600
200,PSPTO_0139-hypothetical protein,PSPTO_0139,hypothetical protein,PSPTO0139
5152,PSPTO_5231-oxidoreductase zinc-binding protein,PSPTO_5231,oxidoreductase zinc-binding protein,PSPTO5231
4307,PSPTO_4361-chromosome initiation inhibitor,PSPTO_4361,chromosome initiation inhibitor,PSPTO4361


## Celda 7: limpiar y filtrar

In [8]:
counts, meta = prep.clean_and_filter(count_matrix, ids_table, selection)
counts.iloc[:5, :5]

Muestras: 15 | Genes antes del filtro: 5618

Tamaño de librería (conteos totales por muestra):
Pto_col_A42     782258
Pto_col_A71    1193944
Pto_col_A66    1253175
Pto_KB_A83     1281080
Pto_col_A74    1319313
Pto_col_A15    1623249
Pto_col_A57    1696944
Pto_col_A25    1788245
Pto_KB_A81     2018229
Pto_col_A13    2131021
Pto_col_A58    2304315
Pto_col_A17    2560461
Pto_col_A59    2994432
Pto_col_A26    3049183
Pto_col_A14    3253487

Genes que pasan el filtro: 5394 de 5618


,Pto_col_A13,Pto_col_A14,Pto_col_A15,Pto_col_A17,Pto_col_A25
PSPTO0001,4167,5779,2727,3160,2859
PSPTO0002,1272,1462,851,1348,935
PSPTO0003,859,1341,580,914,497
PSPTO0004,2645,3430,2186,3062,2109
PSPTO0005,98,184,81,221,128


## Celda 8: añadir GSM y guardar

In [9]:
# anade el codigo GSM a cada muestra
meta = prep.add_gsm(meta, samples)
counts.to_csv(PROC / "counts_filtered.csv")
meta.to_csv(PROC / "sample_metadata.csv")
print(counts.shape, "|", meta["condition"].value_counts().to_dict())
meta

(5394, 15) | {'plant': 13, 'culture': 2}


,condition,gsm
sample,,
Pto_col_A13,plant,GSM2771365
Pto_col_A14,plant,GSM2771381
Pto_col_A15,plant,GSM2771389
Pto_col_A17,plant,GSM2771373
Pto_col_A25,plant,GSM2771401
Pto_col_A26,plant,GSM2771407
Pto_col_A42,plant,GSM2771413
Pto_col_A66,plant,GSM2771451
Pto_col_A57,plant,GSM2771440


## Celda 9: validar accesos del genoma

In [10]:
import annotation as ann

# NCBI devuelve la version vigente de cada acceso
# codigos del cromosoma y los dos plasmidos de DC3000 (sin version)
ACCESSIONS = ["NC_004578", "NC_004633", "NC_004632"]

# consulta, verifica y devuelve codigos con version
accessions = ann.validate_nuccore_accessions(ACCESSIONS)
print(accessions)

NC_004578.1 | Pseudomonas syringae pv. tomato str. DC3000, complete sequence
NC_004633.1 | Pseudomonas syringae pv. tomato str. DC3000 plasmid pDC3000A, complete sequence
NC_004632.1 | Pseudomonas syringae pv. tomato str. DC3000 plasmid pDC3000B, complete sequence
['NC_004578.1', 'NC_004633.1', 'NC_004632.1']


## Celda 10: descargar los GenBank

In [11]:
# para cada acceso: descarga, comprueba y enlista
# si un archivo ya existe en data/raw/, no se descarga otra vez
genbank_paths = [ann.download_genbank(acc, RAW) for acc in accessions]

Descargado: ..\data\raw\NC_004578.1.gb
  NC_004578.1: 6397126 pb, 5731 CDS
Descargado: ..\data\raw\NC_004633.1.gb
  NC_004633.1: 73661 pb, 76 CDS
Descargado: ..\data\raw\NC_004632.1.gb
  NC_004632.1: 67473 pb, 78 CDS
